In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
#setup
import pandas as pd
import numpy as np
import re
import os
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

OPTIONS = ["A", "B", "C", "D", "E"]

print("train:", train.shape, " test:", test.shape)
print(train["answer"].value_counts())

train: (2000, 8)  test: (500, 7)
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [3]:
def map_at_3(true_labels, pred_rankings):
    """Standard MAP@3. pred_rankings: list of ranked-letter-lists per row."""
    scores = []
    for true, preds in zip(true_labels, pred_rankings):
        preds = preds[:3]
        if true not in preds:
            scores.append(0.0)
        else:
            rank = preds.index(true) + 1
            scores.append(1.0 / rank)
    return float(np.mean(scores))

In [4]:
tr, val = train_test_split(train, test_size=0.2, random_state=42, stratify=train["answer"])
print(f"tr: {len(tr)} rows, val: {len(val)} rows")

tr: 1600 rows, val: 400 rows


baseline

In [5]:
freq_order = train["answer"].value_counts().index.tolist()
top3_static = freq_order[:3]
print("Static top-3 by frequency:", top3_static)

baseline_val_preds = [top3_static for _ in range(len(val))]
baseline_score = map_at_3(val["answer"].tolist(), baseline_val_preds)
print(f"Baseline MAP@3 on val: {baseline_score:.4f}")

submission = pd.DataFrame({
    "ID": test["id"].tolist(),
    "Prediction": [" ".join(top3_static) for _ in range(len(test))],
})
submission.to_csv("submission.csv", index=False)
print("Safety-net submission.csv written.")

Static top-3 by frequency: ['B', 'C', 'A']
Baseline MAP@3 on val: 0.4217
Safety-net submission.csv written.


In [6]:
def normalize(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\s]", "", text)
    return text

def build_lookup(df):
    d = df.copy()
    d["norm_prompt"] = d["prompt"].apply(normalize)
    return dict(zip(d["norm_prompt"], d["answer"]))

def exact_match_rank(df, lookup):
    preds, hits = [], 0
    for _, row in df.iterrows():
        key = normalize(row["prompt"])
        if key in lookup:
            ans = lookup[key]
            hits += 1
            rest = [c for c in OPTIONS if c != ans]
            random.shuffle(rest)
            preds.append([ans, rest[0], rest[1]])
        else:
            preds.append(None)
    return preds, hits

In [7]:
tr_lookup = build_lookup(tr)
val_exact_preds, val_hits = exact_match_rank(val, tr_lookup)
print(f"Exact-match hit rate on val: {val_hits}/{len(val)} = {val_hits/len(val):.2%}")

Exact-match hit rate on val: 64/400 = 16.00%


In [8]:
#!pip install wandb -q
#import wandb

#WANDB_ENTITY = "24f2007267-shivani-iit-madras"
#WANDB_PROJECT = "24f2007267-t22026"

#wandb.login()

built fro scratch tfidf 

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def build_pairwise_dataset(df):
    """One row per (question, option) pair, labeled 1 if that option is correct."""
    texts, labels = [], []
    for _, row in df.iterrows():
        for opt in OPTIONS:
            texts.append(row["prompt"] + " [SEP] " + str(row[opt]))
            labels.append(1 if row["answer"] == opt else 0)
    return texts, labels

tr_texts, tr_labels = build_pairwise_dataset(tr)

scratch_vectorizer = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1,2)).fit(tr_texts)
X_tr = scratch_vectorizer.transform(tr_texts)

scratch_clf = LogisticRegression(max_iter=1000, class_weight="balanced", C=0.5)
scratch_clf.fit(X_tr, tr_labels)
print("Scratch model trained:", X_tr.shape)

Scratch model trained: (8000, 11241)


In [10]:
def scratch_rank(df, vectorizer, clf):
    preds = []
    for _, row in df.iterrows():
        texts = [row["prompt"] + " [SEP] " + str(row[opt]) for opt in OPTIONS]
        X = vectorizer.transform(texts)
        probs = clf.predict_proba(X)[:, 1]  # P(this option is correct)
        preds.append([OPTIONS[i] for i in np.argsort(-probs)])
    return preds

val_scratch_preds = scratch_rank(val, scratch_vectorizer, scratch_clf)
scratch_score = map_at_3(val["answer"].tolist(), val_scratch_preds)
top1_scratch = [p[0] for p in val_scratch_preds]
scratch_acc = accuracy_score(val["answer"], top1_scratch)
scratch_f1 = f1_score(val["answer"], top1_scratch, average="macro")
print(f"Model 1 (scratch) — MAP@3: {scratch_score:.4f}, Accuracy: {scratch_acc:.4f}, F1: {scratch_f1:.4f}")

#wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="model1-scratch-tfidf-logreg",
 #          group="model-comparison", job_type="train",
  #         config={"model_type": "from_scratch", "features": "tfidf",
               #    "classifier": "logistic_regression", "max_features": 20000})
#wandb.log({"map@3": scratch_score, "accuracy": scratch_acc, "f1": scratch_f1})
#wandb.finish()

Model 1 (scratch) — MAP@3: 0.9708, Accuracy: 0.9575, F1: 0.9584


## sentence transformers: pretrained model

In [11]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer, util

embed_model = SentenceTransformer("all-mpnet-base-v2")

def embedding_rank(df):
    preds = []
    for _, row in df.iterrows():
        prompt_emb = embed_model.encode(row["prompt"], convert_to_tensor=True)
        opt_embs = embed_model.encode([row[opt] for opt in OPTIONS], convert_to_tensor=True)
        sims = util.cos_sim(prompt_emb, opt_embs).flatten().cpu().numpy()
        preds.append([OPTIONS[i] for i in np.argsort(-sims)])
    return preds

val_emb_preds = embedding_rank(val)
emb_score = map_at_3(val["answer"].tolist(), val_emb_preds)
top1_emb = [p[0] for p in val_emb_preds]
emb_acc = accuracy_score(val["answer"], top1_emb)
emb_f1 = f1_score(val["answer"], top1_emb, average="macro")
print(f"Model 2 (pretrained) — MAP@3: {emb_score:.4f}, Accuracy: {emb_acc:.4f}, F1: {emb_f1:.4f}")

#wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="model2-pretrained-sentence-embeddings",
 #          group="model-comparison", job_type="eval",
  #         config={"model_type": "pretrained", "checkpoint": "all-mpnet-base-v2", "mode": "zero_shot"})
#wandb.log({"map@3": emb_score, "accuracy": emb_acc, "f1": emb_f1})
#wandb.finish()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model 2 (pretrained) — MAP@3: 0.4125, Accuracy: 0.2500, F1: 0.2471


model of additional choice

In [12]:
!pip install peft accelerate -q
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label_map = {o: i for i, o in enumerate(OPTIONS)}
tr_ft, val_ft = tr.copy(), val.copy()
tr_ft["label"]  = tr_ft["answer"].map(label_map)
val_ft["label"] = val_ft["answer"].map(label_map)

def preprocess(examples):
    first  = sum([[p] * 5 for p in examples["prompt"]], [])
    second = sum([[examples[opt][i] for opt in OPTIONS] for i in range(len(examples["prompt"]))], [])
    tok = tokenizer(first, second, truncation=True, padding="max_length", max_length=256)
    return {k: [v[i:i + 5] for i in range(0, len(v), 5)] for k, v in tok.items()}

cols = ["prompt", "A", "B", "C", "D", "E", "label"]
train_ds = Dataset.from_pandas(tr_ft[cols]).map(preprocess, batched=True)
val_ds   = Dataset.from_pandas(val_ft[cols]).map(preprocess, batched=True)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [13]:
base_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8, lora_alpha=16, lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

trainable params: 295,681 || all params: 184,718,594 || trainable%: 0.1601


In [14]:
#wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="model3-lora-deberta-finetune",
 #         group="model-comparison", job_type="train",
  #        config={"model_type": "additional_choice", "base_model": MODEL_NAME, "method": "LoRA",
   #             "r": 8, "lora_alpha": 16, "epochs": 3, "learning_rate": 2e-4})

training_args = TrainingArguments(
    output_dir="/kaggle/working/lora-mcq",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    #report_to="wandb",           # Trainer auto-logs loss/eval curves to the active run
    run_name="model3-lora-deberta-finetune",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

trainer.train()  # <-- confirm this actually executes and produces epoch logs

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,3.197266
2,No log,3.164062
3,No log,3.132812


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=150, training_loss=12.795729166666666, metrics={'train_runtime': 282.5502, 'train_samples_per_second': 16.988, 'train_steps_per_second': 0.531, 'total_flos': 3168260923392000.0, 'train_loss': 12.795729166666666, 'epoch': 3.0})

In [15]:
logits = trainer.predict(val_ds).predictions
top3_idx = np.argsort(-logits, axis=1)[:, :3]
val_clf_preds = [[OPTIONS[i] for i in row] for row in top3_idx]

clf_score = map_at_3(val["answer"].tolist(), val_clf_preds)
top1_clf  = [p[0] for p in val_clf_preds]
clf_acc = accuracy_score(val["answer"], top1_clf)
clf_f1  = f1_score(val["answer"], top1_clf, average="macro")
print(f"Model 3 (additional/LoRA) — MAP@3: {clf_score:.4f}, Accuracy: {clf_acc:.4f}, F1: {clf_f1:.4f}")

#wandb.log({"map@3": clf_score, "accuracy": clf_acc, "f1": clf_f1})
#wandb.finish()

assert clf_score > baseline_score, (
    "Model 3 did not beat the static baseline. Check that every cell in Phase 7 "
    "is type 'Code' (not 'Markdown') and that trainer.train() actually ran to completion."
)
print("Sanity check passed.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Model 3 (additional/LoRA) — MAP@3: 0.5083, Accuracy: 0.3350, F1: 0.3392
Sanity check passed.


model comparision

In [16]:
comparison = pd.DataFrame([
    {"model": "Model 1 — Scratch (TF-IDF + LogReg)",       "accuracy": scratch_acc, "f1": scratch_f1, "map@3": scratch_score},
    {"model": "Model 2 — Pretrained (Sentence-Transformers)", "accuracy": emb_acc,     "f1": emb_f1,     "map@3": emb_score},
    {"model": "Model 3 — Additional (LoRA DeBERTa)",         "accuracy": clf_acc,     "f1": clf_f1,     "map@3": clf_score},
])
print(comparison)

#wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name="model-comparison-summary",
 #          group="model-comparison", job_type="eval")
#wandb.log({"comparison_table": wandb.Table(dataframe=comparison)})
#wandb.finish()

                                          model  accuracy        f1     map@3
0           Model 1 — Scratch (TF-IDF + LogReg)    0.9575  0.958421  0.970833
1  Model 2 — Pretrained (Sentence-Transformers)    0.2500  0.247055  0.412500
2           Model 3 — Additional (LoRA DeBERTa)    0.3350  0.339191  0.508333


bonus llm

In [17]:
!pip install groq -q
import time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq
from kaggle_secrets import UserSecretsClient

client = Groq(api_key=UserSecretsClient().get_secret("GROQ_TOKEN"))

LLM_MODEL   = "openai/gpt-oss-120b"   # llama-3.3-70b-versatile is deprecated on Groq
N_SHOTS     = 5
N_SAMPLES   = 3
MAX_WORKERS = 8

retrieval_vectorizer = TfidfVectorizer().fit(tr["prompt"])
tr_vecs = retrieval_vectorizer.transform(tr["prompt"])

def get_similar_examples(question, ref_df, vec, vec_matrix, n=N_SHOTS):
    q_vec = vec.transform([question])
    sims = cosine_similarity(q_vec, vec_matrix).flatten()
    top_idx = np.argsort(sims)[::-1][:n]
    return ref_df.iloc[top_idx]

def build_prompt(row, examples):
    few_shot = ""
    for _, ex in examples.iterrows():
        few_shot += (f"Question: {ex['prompt']}\n"
                     f"A) {ex['A']} B) {ex['B']} C) {ex['C']} D) {ex['D']} E) {ex['E']}\n"
                     f"Correct answer: {ex['answer']}\n\n")
    return f"""{few_shot}Now answer this:
Question: {row['prompt']}
A) {row['A']}
B) {row['B']}
C) {row['C']}
D) {row['D']}
E) {row['E']}

Reply with ONLY 3 letters separated by spaces, best to least likely. Example: C A B"""

def call_model(prompt, temperature):
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": "You are an expert academic. Use the examples to "
             "guide your answer. Reply with only 3 letters separated by spaces. Nothing else."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=10,
        temperature=temperature,
    )
    reply = resp.choices[0].message.content.strip().upper()
    letters = []
    for c in re.findall(r"\b[ABCDE]\b", reply):
        if c not in letters:
            letters.append(c)
    return letters

def call_with_retry(prompt, temperature, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            return call_model(prompt, temperature)
        except Exception:
            time.sleep(min(30, 2 ** attempt) + random.uniform(0, 1))
    return []

def rank_by_votes(vote_lists):
    scores = {}
    for letters in vote_lists:
        for rank, c in enumerate(letters):
            scores[c] = scores.get(c, 0) + (3 - rank)
    return sorted(OPTIONS, key=lambda c: -scores.get(c, 0))

def llm_predict_row(row, ref_df, vec, vec_matrix):
    examples = get_similar_examples(row["prompt"], ref_df, vec, vec_matrix)
    prompt = build_prompt(row, examples)
    votes = []
    for i in range(N_SAMPLES):
        temp = 0.0 if i == 0 else 0.6
        letters = call_with_retry(prompt, temp)
        if letters:
            votes.append(letters)
    return rank_by_votes(votes)[:3] if votes else ["B", "A", "C"]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.6 MB/s eta 0:00:00


val_rows = list(val.iterrows())
val_llm_preds = [None] * len(val_rows)

def worker(i, row):
    return i, llm_predict_row(row, tr, retrieval_vectorizer, tr_vecs)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(worker, i, row): i for i, (_, row) in enumerate(val_rows)}
    for future in as_completed(futures):
        i, pred = future.result()
        val_llm_preds[i] = pred

llm_score = map_at_3(val["answer"].tolist(), val_llm_preds)
print(f"LLM ensemble MAP@3 on val: {llm_score:.4f}")

In [18]:
MAX_WORKERS = 64  # bump way up, this is I/O-bound not CPU-bound

val_rows = list(val.iterrows())
val_llm_preds = [None] * len(val_rows)

def worker(i, row):
    return i, llm_predict_row(row, tr, retrieval_vectorizer, tr_vecs)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(worker, i, row): i for i, (_, row) in enumerate(val_rows)}
    for future in as_completed(futures):
        i, pred = future.result()
        val_llm_preds[i] = pred

llm_score = map_at_3(val["answer"].tolist(), val_llm_preds)
print(f"LLM ensemble MAP@3 on val: {llm_score:.4f}")

LLM ensemble MAP@3 on val: 0.4142


In [19]:
def to_points(ranked_list):
    points = {o: 0.0 for o in OPTIONS}
    for rank, letter in enumerate(ranked_list):
        points[letter] = 5 - rank
    return points

def fuse_with_override(pred_sets, weights, override_preds):
    n_rows = len(override_preds)
    fused = []
    for i in range(n_rows):
        if override_preds[i] is not None:
            fused.append(override_preds[i])
            continue
        combined = {o: 0.0 for o in OPTIONS}
        for preds, w in zip(pred_sets, weights):
            pts = to_points(preds[i])
            for o in OPTIONS:
                combined[o] += w * pts[o]
        fused.append(sorted(OPTIONS, key=lambda o: -combined[o])[:3])
    return fused

model_names  = ["model3_lora", "llm", "model2_embeddings", "model1_scratch"]
model_preds  = [val_clf_preds, val_llm_preds, val_emb_preds, val_scratch_preds]
model_scores = [clf_score, llm_score, emb_score, scratch_score]
print(dict(zip(model_names, model_scores)))

import itertools
weight_grid = [0.0, 0.5, 1.0, 1.5, 2.0]
best_score, best_weights = 0.0, None

for w in itertools.product(weight_grid, repeat=len(model_preds)):
    if sum(w) == 0:
        continue
    fused = fuse_with_override(model_preds, w, val_exact_preds)
    score = map_at_3(val["answer"].tolist(), fused)
    if score > best_score:
        best_score, best_weights = score, w

print(f"Best fused MAP@3 on val: {best_score:.4f}")
print("Best weights:", dict(zip(model_names, best_weights)))

In [20]:
from sklearn.model_selection import StratifiedKFold
import itertools

model_names  = ["model3_lora", "llm", "model2_embeddings", "model1_scratch"]
model_preds  = [val_clf_preds, val_llm_preds, val_emb_preds, val_scratch_preds]
model_scores = [clf_score, llm_score, emb_score, scratch_score]
print(dict(zip(model_names, model_scores)))

true_labels = val["answer"].tolist()
weight_grid = [0.0, 0.5, 1.0, 1.5, 2.0]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(val, val["answer"]))

best_score, best_weights = 0.0, None
for w in itertools.product(weight_grid, repeat=len(model_preds)):
    if sum(w) == 0:
        continue
    fold_scores = []
    for _, fold_idx in folds:
        sub_true = [true_labels[i] for i in fold_idx]
        #sub_preds = [model_preds[m][i] for m in range(len(model_preds))]  # placeholder, fixed below
        # build per-fold pred_sets and overrides sliced to fold_idx
        pred_sets_fold = [[preds[i] for i in fold_idx] for preds in model_preds]
        override_fold  = [val_exact_preds[i] for i in fold_idx]
        fused = fuse_with_override(pred_sets_fold, w, override_fold)
        fold_scores.append(map_at_3(sub_true, fused))
    avg_score = np.mean(fold_scores)
    if avg_score > best_score:
        best_score, best_weights = avg_score, w

print(f"Best CV-averaged MAP@3: {best_score:.4f}")
print("Best weights:", dict(zip(model_names, best_weights)))

{'model3_lora': 0.5083333333333333, 'llm': 0.4141666666666666, 'model2_embeddings': 0.4125, 'model1_scratch': 0.9708333333333334}
Best CV-averaged MAP@3: 0.9738
Best weights: {'model3_lora': 0.0, 'llm': 0.0, 'model2_embeddings': 0.0, 'model1_scratch': 0.5}


final test predictions and submission csv

In [21]:
full_lookup = build_lookup(train)
test_exact_preds, test_hits = exact_match_rank(test, full_lookup)
print(f"Exact-match hit rate on test: {test_hits}/{len(test)} = {test_hits/len(test):.2%}")

retrieval_vectorizer_full = TfidfVectorizer().fit(train["prompt"])
train_vecs_full = retrieval_vectorizer_full.transform(train["prompt"])

Exact-match hit rate on test: 268/500 = 53.60%


wandb.init(entity = "WANDB_ENTITY",project="WANDB_PROJECT", name="model3-lora-test-eval")

test_logits = trainer.predict(test_ds).predictions

# Model 3 (LoRA) on test
test_ds = Dataset.from_pandas(test[["prompt", "A", "B", "C", "D", "E"]]).map(preprocess, batched=True)
test_logits = trainer.predict(test_ds).predictions
test_top3_idx = np.argsort(-test_logits, axis=1)[:, :3]
test_clf_preds = [[OPTIONS[i] for i in row] for row in test_top3_idx]

# Model 2 (embeddings) on test
test_emb_preds = embedding_rank(test)

# Model 1 (scratch) on test
test_scratch_preds = scratch_rank(test, scratch_vectorizer, scratch_clf)

print("Model 1/2/3 test predictions ready.")

In [22]:
from transformers.integrations import WandbCallback

trainer.remove_callback(WandbCallback)

# Model 3 (LoRA) on test
test_ds = Dataset.from_pandas(test[["prompt", "A", "B", "C", "D", "E"]]).map(preprocess, batched=True)
test_logits = trainer.predict(test_ds).predictions
test_top3_idx = np.argsort(-test_logits, axis=1)[:, :3]
test_clf_preds = [[OPTIONS[i] for i in row] for row in test_top3_idx]
# Model 2 (embeddings) on test
test_emb_preds = embedding_rank(test)
# Model 1 (scratch) on test
test_scratch_preds = scratch_rank(test, scratch_vectorizer, scratch_clf)
print("Model 1/2/3 test predictions ready.")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Model 1/2/3 test predictions ready.


In [23]:
# LLM predictions on test — parallel, with progress logging
test_llm_preds = [None] * len(test)
test_rows = list(test.iterrows())

def worker_test(i, row):
    return i, llm_predict_row(row, train, retrieval_vectorizer_full, train_vecs_full)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(worker_test, i, row): i for i, (_, row) in enumerate(test_rows)}
    done = 0
    for future in as_completed(futures):
        i, pred = future.result()
        test_llm_preds[i] = pred
        done += 1
        if done % 20 == 0:
            print(f"{done}/{len(test)} LLM predictions done")

print("LLM test predictions ready.")

20/500 LLM predictions done
40/500 LLM predictions done
60/500 LLM predictions done
80/500 LLM predictions done
100/500 LLM predictions done
120/500 LLM predictions done
140/500 LLM predictions done
160/500 LLM predictions done
180/500 LLM predictions done
200/500 LLM predictions done
220/500 LLM predictions done
240/500 LLM predictions done
260/500 LLM predictions done
280/500 LLM predictions done
300/500 LLM predictions done
320/500 LLM predictions done
340/500 LLM predictions done
360/500 LLM predictions done
380/500 LLM predictions done
400/500 LLM predictions done
420/500 LLM predictions done
440/500 LLM predictions done
460/500 LLM predictions done
480/500 LLM predictions done
500/500 LLM predictions done
LLM test predictions ready.


In [24]:
# Fuse everything using the best weights found in Phase 10 — same order as model_names there
test_model_preds = [test_clf_preds, test_llm_preds, test_emb_preds, test_scratch_preds]
final_preds = fuse_with_override(test_model_preds, best_weights, test_exact_preds)

submission = pd.DataFrame({
    "ID": test["id"].tolist(),
    "Prediction": [" ".join(p) for p in final_preds],
})

assert len(submission) == len(test), "Row count mismatch"
assert submission["Prediction"].apply(lambda p: len(p.split()) == 3).all(), "Some rows don't have 3 letters"
assert submission["ID"].is_unique, "Duplicate IDs"

submission.to_csv("submission.csv", index=False)
print(f"Final submission.csv written — {len(submission)} rows.")
submission.head()

Final submission.csv written — 500 rows.


,ID,Prediction
0,1,A C D
1,2,B D C
2,3,B E D
3,4,E A B
4,5,C E B
